In [ ]:
# DATASET:

# MODEL:

# Risolvere il problema del joinare 2 fasi divise da un checkpoint (alla fine di tutto, elimini tutti i passi che hanno il delta time stamp = 0)

# TRAINING:

# ESECUZIONE DI TUTTI I MODELLI SU TUTTI I DATASET

# ELABORAZIONE DEI RISULTATI (CON ESTENSIONE SULLE METRICHE)

# Libraries

In [2]:
#%pip install lark graphviz

In [3]:
import pmlb
import pandas as pd
import numpy as np
import time
import os
import yaml
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import itertools
import json

import LearningStrategy as ls
from Model import SymbolicRegression
import Operations as op
import Errors as er
import Indicators as ind
import InformationSOTA.artificial_formulas as af

# Prepare datasets

In [29]:
cavalab_bb_results = pd.read_feather('InformationSOTA/srbench-master/results/black-box_results.feather')
cavalab_gt_results = pd.read_feather('InformationSOTA/srbench-master/results/ground-truth_results.feather')

bb_dataset_names = [i for i in cavalab_bb_results["dataset"].unique()]
gt_dataset_names = [i for i in cavalab_gt_results["dataset"].unique()]

for i in bb_dataset_names + gt_dataset_names: print(i)

1027_ESL
1028_SWD
1029_LEV
1030_ERA
1089_USCrime
1096_FacultySalaries
1191_BNG_pbc
1193_BNG_lowbwt
1196_BNG_pharynx
1199_BNG_echoMonths
1201_BNG_breastTumor
1203_BNG_pwLinear
1595_poker
192_vineyard
195_auto_price
197_cpu_act
201_pol
207_autoPrice
210_cloud
215_2dplanes
218_house_8L
225_puma8NH
227_cpu_small
228_elusage
229_pwLinear
230_machine_cpu
294_satellite_image
344_mv
4544_GeographicalOriginalofMusic
485_analcatdata_vehicle
503_wind
505_tecator
519_vinnie
522_pm10
523_analcatdata_neavote
527_analcatdata_election2000
529_pollen
537_houses
542_pollution
547_no2
556_analcatdata_apnea2
557_analcatdata_apnea1
560_bodyfat
561_cpu
562_cpu_small
564_fried
573_cpu_act
574_house_16H
579_fri_c0_250_5
581_fri_c3_500_25
582_fri_c1_500_25
583_fri_c1_1000_50
584_fri_c4_500_25
586_fri_c3_1000_25
588_fri_c4_1000_100
589_fri_c2_1000_25
590_fri_c0_1000_50
591_fri_c1_100_10
592_fri_c4_1000_25
593_fri_c1_1000_10
594_fri_c2_100_5
595_fri_c0_1000_10
596_fri_c2_250_5
597_fri_c2_500_5
598_fri_c0_1000_25

In [ ]:
#Calculate the seeds used in every datasets

dict_seeds = {}

for name in bb_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_bb_results.loc[cavalab_bb_results["dataset"] == name, "random_state"].unique()])
    
for name in gt_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_gt_results.loc[cavalab_gt_results["dataset"] == name, "random_state"].unique()])
    
#for key in dict_seeds.keys(): print(key, dict_seeds[key])

total_set = set([x for x in cavalab_bb_results["random_state"].unique()]).union(set([x for x in cavalab_gt_results["random_state"].unique()]))

for key in dict_seeds.keys():
    
    total_set = total_set.intersection(dict_seeds[key])
    
print(total_set)


{15795, 860, 23654, 5390}


In [48]:
def split(x, perc):
    train_len = int(perc*len(x))
    return x[:train_len], x[train_len:]

def is_prefix(name, prefix):
    return name[:len(prefix)] == prefix

def adapt_name(name):
    
    expection_names = {
        "feynman_I_48_2": "feynman_I_48_20",
                       }
    
    if(name in expection_names.keys()): name = expection_names[name]
    
    if(is_prefix(name, "feynman_test")): return name[len('feynamn_'):]
    elif(is_prefix(name, "feynman")): return name[len('feynamn_'):].replace('_', '.')
    elif(is_prefix(name, "strogatz")): return name.replace('strogatz', 'd')



#Si ridà direttamente il dataset splittato: si fa uno split hold out del 75 di training set e 25% di test set (lo dice alla fine della sezione 3.2 del paper Call for action, tipo preprint).
#Da capire il seed quali sono.
def get_dataset(name):
    
    if(is_prefix(name, "art_f_")):
        
        try:
        
            X = np.linspace(-100, 100, 10000)
            formula = af.artificial_formulas[int(name[len("art_f_"):])]
            Y = eval(formula)
            if(type(Y) in [int]): Y = np.array([Y]*X.shape[0])
            
            mask = np.isfinite(Y)

            X = X[mask]
            Y = Y[mask]
   
            result = {"x_dict": {"X": X}, "y": Y, "formula": formula}
            
        except IndexError: raise ValueError(f"Dataset {name} are not aviable")

    elif(name in bb_dataset_names):
    
        x, y = pmlb.fetch_data(name, return_X_y=True)
        
        x = x.T

        result = {"x_dict": {f"X{i}": x[i] for i in range(x.shape[0])}, "y": y, "formula": None}
    
    elif(name in gt_dataset_names):
        
        adapted_name = adapt_name(name)
        
        formulas = pd.read_csv("InformationSOTA/Equations.csv")
        
        try: formula_founded = formulas.loc[formulas["Filename"] == adapted_name, "Formula"].tolist()[0]
        except IndexError: formula_founded = None
        
        if(is_prefix(name, "feynman_test")): file = f"InformationSOTA/Feynman_bonus_with_units/{adapted_name}"
        elif(is_prefix(name, "feynman")): file = f"InformationSOTA/Feynman_with_units/{adapted_name}"
        elif(is_prefix(name, "strogatz")): file = f"InformationSOTA/ode-strogatz-master/{adapted_name}.txt"
        
        if(is_prefix(name, "feynman")):
            
            feature_names = [x["name"] for x in yaml.safe_load(open(f"InformationSOTA/pmlb-master/datasets/{name}/metadata.yaml", "r"))["features"]]
            
            table = np.loadtxt(file).T
            result = {"x_dict": {name: table[i] for name, i in zip(feature_names, range(table.shape[0]-1))}, "y": table[-1], "formula": formula_founded}
        
        else:
            
            table = pd.read_csv(file)
            result = {"x_dict": {l: table[l].to_numpy() for l in table.columns.tolist() if l != "label"}, "y": table["label"].to_numpy(), "formula": formula_founded}
            
        
    else: raise ValueError(f"Dataset {name} are not aviable")
     
    if(np.any(~np.isfinite(result["y"])) or any(np.any(~np.isfinite(result["x_dict"][k])) for k in result["x_dict"].keys())):
        raise ValueError("Some data are not finite value, the dataset is not valid")
    
    
    #Prepare the hyper-parameters choosed for this experiments
    
    shuffle_seed = 860  #Form the 5 universal seed, we choose 860
    perc_split = 0.75
    
    #Prepare the slots for split
    result["x_dict train"] = {}
    result["x_dict test"] = {}
    
    #Prepare the shuffler
    shuffler = np.random.default_rng(seed = shuffle_seed)
    
    #Split all X
    for key in result["x_dict"].keys():
        shuffler.shuffle(result["x_dict"][key])
        result["x_dict train"][key], result["x_dict test"][key] = split(result["x_dict"][key], perc_split)
    
    #Split Y
    shuffler.shuffle(result["y"])
    result["y train"], result["y test"] = split(result["y"], perc_split)
        
    #Delete old Y and all X
    del result["x_dict"]
    del result["y"]

    return result

for name in ["227_cpu_small",
             "feynman_I_39_22",
             "feynman_test_6",
             "strogatz_glider2",
             "art_f_0"]:
    
    print(f"name: {name}")
    
    dataset = get_dataset(name)
    
    print(f"formula: {dataset['formula']}")
    print("x_dict train:")
    for key in dataset["x_dict train"].keys(): print(key, dataset["x_dict train"][key].shape)
    print("x_dict test:")
    for key in dataset["x_dict test"].keys(): print(key, dataset["x_dict test"][key].shape)

    print(f"y train: {dataset['y train'].shape}")
    print(f"y test: {dataset['y test'].shape}")

name: 227_cpu_small
formula: None
x_dict train:
X0 (6144,)
X1 (6144,)
X2 (6144,)
X3 (6144,)
X4 (6144,)
X5 (6144,)
X6 (6144,)
X7 (6144,)
X8 (6144,)
X9 (6144,)
X10 (6144,)
X11 (6144,)
x_dict test:
X0 (2048,)
X1 (2048,)
X2 (2048,)
X3 (2048,)
X4 (2048,)
X5 (2048,)
X6 (2048,)
X7 (2048,)
X8 (2048,)
X9 (2048,)
X10 (2048,)
X11 (2048,)
y train: (6144,)
y test: (2048,)
name: feynman_I_39_22
formula: n*kb*T/V
x_dict train:
n (750000,)
T (750000,)
V (750000,)
kb (750000,)
x_dict test:
n (250000,)
T (250000,)
V (250000,)
kb (250000,)
y train: (750000,)
y test: (250000,)
name: feynman_test_6
formula: sqrt(1+2*epsilon**2*E_n*L**2/(m*(Z_1*Z_2*q**2)**2))
x_dict train:
epsilon (750000,)
L (750000,)
m (750000,)
Z_1 (750000,)
Z_2 (750000,)
q (750000,)
E_n (750000,)
x_dict test:
epsilon (250000,)
L (250000,)
m (250000,)
Z_1 (250000,)
Z_2 (250000,)
q (250000,)
E_n (250000,)
y train: (750000,)
y test: (250000,)
name: strogatz_glider2
formula: x - cos(y)/x
x_dict train:
x (300,)
y (300,)
x_dict test:
x (100,)

# Prepare models

## Prepare various models

In [ ]:
def indicators_package_extensions(nodes, unary_operations, TR_inputs_dict):
    
    combinations = default_package_extensions(nodes, unary_operations, TR_inputs_dict)[0]
    
    result = []
    
    analyzed_nodes = [{"node": x, 
                       "constant": 1 / (np.var(x.get_memory("backward")) + 1),
                       "pearson": ind.pearson_correlation(x.get_memory("forward"), x.get_memory("backward")),
                       "chaterjee": ind.chatterjee_correlation(x.get_memory("forward"), x.get_memory("backward"))} for x in nodes]
    
    #Consider the node like: the linear distance correlation between output and excepted output (explain this), and if 2 nodes are with same distance linear correlation.
    #considering the node with more potential construction with chaterjee (explain this).
    for s in sorted(analyzed_nodes, key = lambda x: ((np.abs(x["pearson"]) if not x["node"].is_constant() else x["constant"]), -np.abs(x["chaterjee"]))):
        result.append([x for x in combinations if x["old node"] is s["node"]])

    return result



#Foreach node:

#Nodo interno: ou
#foglia feature : f + ou
#foglia const   : f+1 + ou

# Nodo interno: aggiungo tutte le varie possibili operazioni sopra
# Foglia: aggiungo tutte le feature + costante, non ci metto la combinazione di operazione unaria + feature per osservare il comportamento del modello senza provare a 
# scavalcare con dei pattern già complessi (perchè sennò bisognava provare anche con pattern complessi di 2 livello, 3 livello etc.), e da lì poi l' esplosione,
# quindi per vedere se il modello riesce a catturare le relazioni strutturali giuste senza fare i salti


def default_package_extensions(nodes, unary_operations, TR_inputs_dict):

    def get_constant(memory):
        
        if(isinstance(memory, np.ndarray)):
            mask = memory[np.isfinite(memory)]
            if(len(mask) == 0): raise ValueError("It is not possible to do the mean with empty list")

        return np.mean(memory[np.isfinite(memory)]) if isinstance(memory, np.ndarray) else memory
    
    combinations = []
    
    #n nodes
    for n in nodes:
        
        #f or f + 1 (about leaf cost)
        if(isinstance(n, op.Leaf)):
            
            leafs = [op.Leaf(x) for x in TR_inputs_dict.keys() if x != n.get_name()]
            leafs.append(op.Leaf(get_constant(n.get_memory("backward"))))
                
            for l in leafs: combinations.append({"old node": n, "new node": l})
        
        
        #ou (add unary operation)
        for ou in unary_operations:
            
            oper = ou()
            
            if(not oper.check_input_domain(0, n.get_memory("forward"))): continue
            
            if(not n.is_constant()):
            
                leaf = n.copy_sub_tree()

                oper.new_connection(leaf, 0)
                combinations.append({"old node": n, "new node": oper.copy_sub_tree()})
                oper.free_connection(0)


    for i in range(len(combinations)):
        
        combinations[i]["new node"].forward_calculation_memory_sub_tree(TR_inputs_dict).set_memory("backward", combinations[i]["old node"].get_memory("backward"))
        

    #print("COMBINATIONS" + '-'*100)
    #for i in combinations: print(i)
    
    #raise ValueError("STOP")
    
    return [combinations]



# Prepare tools for trainings

In [ ]:
#State checkpoint: metrics, extension dictionary, counter
def save_checkpoint(label, metrics, memory_extension, c):
    json.dump({"metrics": metrics, "memory_extension": memory_extension, "counter": c},
              open(f"Results/Checkpoints/{label}.json", "w", encoding = "utf-8"),
              indent = 4,
              ensure_ascii = False)

def load_checkpoint(label):
    if(not os.path.exists(f"Results/Checkpoints/{label}.json")): return None
    return json.load(open(f"Results/Checkpoints/{label}.json", "r", encoding="utf-8"))


class Counter:
    
    def __init__(self, i, r):
        self.__i = i
        self.__r = r
        
    def __call__(self):
        self.__r += 1
        return self.__r >= self.__i
    
    def get_count(self):
        return self.__r
    

class DeltaTimeCalculator:
    
    def __init__(self):
        self.__last_timestamp = None
        
    def __call__(self, timestamp):
        
        result = timestamp - self.__last_timestamp if self.__last_timestamp != None else 0
        self.__last_timestamp = timestamp
        return result


step_checkpoint = 500

def stop_criteria_function(strategy_learning, counter, TR_output, label):
    
    model = strategy_learning.get_model()
    
    go = counter() or er.MAE(model.get_root().get_memory("forward"), TR_output) <= 1e-6
    
    c = counter.get_count()
    
    if(c % step_checkpoint == 0):
        
        metrics = strategy_learning.get_metrics()
        memory_extension = strategy_learning.get_dict_extension_history()
        
        save_checkpoint(label, metrics, memory_extension, c)
        
        print(f"{label}, step {c}, checkpoint done!")
    
    return go

## Prepare models

In [ ]:
def total_confidence_leafs(node):
    
    def f(node):
    
        result = []
        for c in node.get_childrens(): result += f(c)
        return result + [node.calculation_hat_confidence()] if isinstance(node, op.Leaf) else result
    
    result = f(node)
    
    return np.mean(result)

def root_metric(model, metric):
    root = model.get_root()
    return metric(root.get_memory("forward"), root.get_memory("backward"))



param_guides = [lambda model: root_metric(model, ind.kendall_correlation),
                lambda model: root_metric(model, ind.spearman_correlation),
                lambda model: root_metric(model, ind.chatterjee_correlation),
                lambda model: root_metric(model, er.MAE),
                lambda model: total_confidence_leafs(model)]

strategies_package_nodes = [default_package_extensions,
                            indicators_package_extensions]

strategies_filter_nodes = [lambda x: True,
                           lambda x: isinstance(x, op.Leaf)]



def execute(c):
  
    #Extract params
    param_guide = param_guides[c['i']]
    strategy_package = strategies_package_nodes[c['r']]
    strategy_filter = strategies_filter_nodes[c['k']]
    dataset_name = c['d']
    
    label = f"version_{c['i']}_{c['r']}_{c['k']}_{c['d']}"
    
    #If experiment is already done, do not the experiment
    if(os.path.exists(f"Results/RoughResults/{label}.json")): return

    model = SymbolicRegression.string_to_tree("0")
    
    #Extract inputs and output accordind to the dataset name
    #dataset = get_dataset(dataset_name)
    #TR_inputs_train = dataset["x_dict train"]
    #TR_output_train = dataset["y train"]
    
    TR_inputs_train = {}
    TR_output_train = np.array([1, 1, 1])
    
    learning_strategy = ls.AnalysisRamification([op.neg, op.div, op.abs, op.exp, op.ln, op.sin, op.arcsin],
                                                    lambda candidates: candidates,  #Filter phase
                                                    lambda candidates: sorted(candidates, key = lambda x: -x["delta heuristic metrics"]["Param"]),  #Sort phase
                                                    lambda n, uo, TR_inputs: strategy_package([x for x in n if strategy_filter(x)], uo, TR_inputs),                                                
                                                    {"Param": param_guide})
    
    #Set the value of last checkpoint if exists
    checkpoint_loaded = load_checkpoint(label)
    
    delta_timer = DeltaTimeCalculator()
    
    steps = 10000
    
    if(checkpoint_loaded != None):
        
        learning_strategy.set_metrics(checkpoint_loaded["metrics"])
        learning_strategy.set_dict_extension_history(checkpoint_loaded["memory_extension"])
        counter = Counter(steps, checkpoint_loaded["counter"])
        
    else:
        
        counter = Counter(steps, 0)

    metric_results = model.learn(TR_inputs_train,
                                TR_output_train,
                                learning_strategy,
                                stop_criteria = lambda ls: stop_criteria_function(ls, counter, TR_output_train, label),
                                metric_functions = {"tree": lambda model: str(model), "timestamp": lambda model: delta_timer(time.time())})
    
    #Save the metrics in the folder
    json.dump(metric_results, open(f"Results/RoughResults/{label}.json", "w", encoding = "utf-8"), indent = 4, ensure_ascii = False)
    #if(os.path.exists(f"Results/Checkpoints/{label}.json")): os.remove(f"Results/Checkpoints/{label}.json")



tasks = []

for i, r, k, d in itertools.product(
        range(len(param_guides)),
        range(len(strategies_package_nodes)),
        range(len(strategies_filter_nodes)),
        [f"art_f_{x}" for x in range(len(af.artificial_formulas))] + bb_dataset_names + gt_dataset_names
    ):
    
    tasks.append({'i': i, 'r': r, 'k': k, 'd': d})


# Experiments

In [12]:
print(f"Experiments done: {len(os.listdir('Results/RoughResults'))} / {len(tasks)}")

execute(tasks[0])

#executor = ThreadPoolExecutor(max_workers = 1)

#executor.map(execute, tasks)

Experiments done: 0 / 5240
